In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get("HF_TOKEN"))

In [ ]:
!pip install -U torchao

In [ ]:
!pip uninstall -y pillow
!pip install pillow==10.4.0

In [ ]:
!pip install -U langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu

In [ ]:
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
from pdf2image import convert_from_path
import pytesseract
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

test_context = []
pdf = "/content/drive/MyDrive/consulting_agreementpdf.pdf"
def OCR1(pdf):
  pages = convert_from_path(pdf,dpi=300)
  for i,pagei in enumerate(pages):
    ft = pytesseract.image_to_string(pagei)
    test_context.append(f"--- PAGE {i + 1} ---\n{ft}\n")
  return'\n'.join(test_context)

test_context = OCR1(pdf)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=250,
    length_function=len
)
chunks = text_splitter.split_text(test_context)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_texts(chunks, embeddings)
retriever = db.as_retriever(
    search_kwargs={"k": 5}
)

# Model path
MODEL_PATH = "/content/drive/MyDrive/Fine Tuning/raft_qlora_outputv212/final_qlora_model"

print("Loading QLoRA model...")

# Load model
model = AutoPeftModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

print("✓ Model Loaded Successfully")

questions = [
"What is the CONSULTANT company in this Agreement?",
"When is the agreement dated?",
"What does this Agreement establish between the Company and the Executive?",
"What is the term of this Agreement?",
"What are the key responsibilities?",
"What is the payment or consideration?",
"What consulting fee is payable to the Consultant?",
"Can the agreement be renewed?",
"Under what conditions can the agreement be terminated?",
"Does this Agreement contain a confidentiality clause?",
"Is there a non-compete clause?",
"What is the governing law?",
"How will disputes be resolved?",
"Who owns the intellectual property?",
"What are the key obligations of each party?"]
gen_ans = []
for test_question in questions:
  match1 = retriever.invoke(test_question)
  ret_context = "\n\n".join(
    [doc.page_content for doc in match1]
)
  prompt = f"""[INST]
  Based on the following contract information, answer the question.

  Answer ONLY using information explicitly present in the contract.

  Contract Information:
  {ret_context}

  Question:
  {test_question}
  [/INST]"""

  # Tokenize
  inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

  # Generate Answer
  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=60,
          temperature=0.0,
          do_sample=False,
          repetition_penalty=1.1,
          pad_token_id=tokenizer.eos_token_id,
      )

  # Decode
  generated_text = tokenizer.decode(
      outputs[0][inputs["input_ids"].shape[1]:],
      skip_special_tokens=True
  ).strip()
  gen_ans.append(generated_text)
  print("\nQuestion:", test_question)
  print("Generated Answer:", generated_text)
df = pd.DataFrame({"Questions":questions, "Answers":gen_ans})
df.to_csv("Legal_QA_Results.csv", index=False)

In [ ]:
!pip install -U transformers==4.52.4 peft==0.19.1 accelerate bitsandbytes

In [ ]:
!apt-get update -qq
!apt-get install -y poppler-utils

In [ ]:
!pip install pytesseract pdf2image Pillow
